In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Literal 
# set root direcotry to bluedot-faithfulness
import os
project_root = Path.cwd().parents[1]   
sys.path.insert(0, str(project_root))

from src.templates.heart_disease import HeartDisease


In [2]:
# classes
@dataclass
class HammingBall:
    """Represents a hamming group - a set of similar data points"""
    group_id: int
    reference_idx: int  # Index of the reference point
    counterfactual_indices: List[int]  # All counterfactual indices including reference
    max_distance: int  # Maximum hamming distance from reference
    feature_names: List[str]  # Names of features that vary
    
    def to_dict(self):
        return asdict(self)
    
# x = HammingBall(1,1,[1,2,3,4,5], 2,["name", "name2"]).to_dict()
# x



In [4]:
def compute_all_pairwise_distances(df, feature_cols, 
                                   max_distance) :
    """
    Compute all pairwise hamming distances using vectorized operations.
    Much faster than computing distances one at a time.
    
    Args:
        df: DataFrame containing the data
        feature_cols: List of feature column names to compare
        max_distance: If provided, can enable early stopping optimizations
        
    Returns:
        n×n numpy array where distances[i,j] = hamming distance between rows i and j
    
    returns np.ndarray 
    size = (len(ds), len(ds))
    """
    # Convert to numpy array for vectorized operations
    data = df[feature_cols].values
    n = len(data)
    
    # Compute all pairwise differences at once using broadcasting
    # Shape: (n, n, num_features)
    # This is memory intensive but very fast
    differences = data[:, np.newaxis, :] != data[np.newaxis, :, :]
    
    # Sum along feature axis to get distances
    # Shape: (n, n)
    distances = differences.sum(axis=2)
    
    return distances

def build_neighbor_graph(df, feature_cols, max_distance):
    """
    Pre-compute neighbor graph for all points.
    This is O(n²) but only done once, making subsequent operations much faster.
    
    Args:
        df: DataFrame containing the data
        feature_cols: List of feature column names
        max_distance: Maximum hamming distance to be considered neighbors
        
    Returns:
        Dictionary mapping each index to its set of neighbors (including itself)
    """
    print(f"Pre-computing neighbor graph for {len(df)} points...")
    
    # Compute all distances at once - O(n²) but vectorized
    distances = compute_all_pairwise_distances(df, feature_cols, max_distance)
    
    # Build neighbor graph
    neighbor_graph = {}
    
    for i in range(len(df)):
        # Find all points within max_distance (including self)
        neighbors = set(np.where(distances[i] <= max_distance)[0])
        neighbor_graph[i] = neighbors
    
    print(f"Neighbor graph built. Average neighbors per point: "
          f"{np.mean([len(neighbors) for neighbors in neighbor_graph.values()]):.1f}")
    
    return neighbor_graph


def identify_varying_features(df: pd.DataFrame, indices: List[int], 
                              feature_cols: List[str]) -> List[str]:
    """
    Identify which features vary within a hamming group
    
    Args:
        df: DataFrame containing the data
        indices: List of row indices in the group
        feature_cols: List of feature column names
        
    Returns:
        List of feature names that vary
    """
    if len(indices) <= 1:
        return []
    
    varying_features = []
    for col in feature_cols:
        values = df.iloc[indices][col].unique()
        if len(values) > 1:
            varying_features.append(col)
    
    return varying_features


In [7]:
# converter class

class TabularToTextConverter:
    """
    Converts tabular datasets to text datasets for counterfactual intervention
    """

    def __init__(self,
            df,
            target_col,
            description_generator,
            prompt_generator,
            dataset_name: str = "unknown",
            target_formatter = None):
        """
        Initialize converter with a pandas DataFrame
        
        Args:
            df: Input DataFrame with categorical features
            target_col: Name of the target/label column (if any)
            description_generator: Function to generate text descriptions from rows.
                                    Should have signature: (row_idx, row_data, feature_cols) -> str
                                    If None, uses a simple default description.
            dataset_name: Name of the dataset for schema records
            target_formatter: Optional function to format target values (e.g., 0/1 -> "YES"/"NO")
                            If None, uses str() conversion. Should accept int and return str.
        """
        self.df = df.copy()
        self.target_col = target_col
        self.dataset_name = dataset_name
        self.hamming_groups = []
        self.description_generator = description_generator or self._default_description
        self.prompt_generator = prompt_generator
        self.target_formatter = target_formatter or str
        
        # Separate features from target
        if target_col and target_col in df.columns:
            self.feature_cols = [col for col in df.columns if col != target_col]
        else:
            self.feature_cols = list(df.columns)
        
        print(f"Initialized with {len(df)} rows and {len(self.feature_cols)} features")
        print(f"Features: {self.feature_cols}")

    def find_hamming_balls_repeated(self, max_distance: int = 3,
                                   min_group_size: int = 5,
                                   max_group_size: int = 15,
                                   epsilon: float = 0.2) -> List[HammingBall]:
        """
        Find hamming balls centered at every data point (allows repeated entries).
        
        For each data point, attempts to find a balanced subset of neighbors where:
        - All neighbors are within max_distance hamming distance
        - Subset has at least min_group_size points
        - Balance parameter = |same_target_count - diff_target_count| / group_size < epsilon
        
        Uses a greedy approach with randomness:
        - Start with center point
        - Alternate between adding points with same/different target values
        - Randomly select from available points with needed target value
        - Keep growing until balance >= epsilon or max_group_size reached
        
        Args:
            max_distance: Maximum hamming distance for neighbors
            min_group_size: Minimum size of each hamming ball
            max_group_size: Maximum size of each hamming ball
            epsilon: Maximum allowed balance parameter (0=perfectly balanced, 1=completely imbalanced)
            
        Returns:
            List of HammingBall objects (one per valid center point)
        """
        print(f"\nFinding repeated hamming balls (max_distance={max_distance}, "
              f"min_size={min_group_size}, max_size={max_group_size}, epsilon={epsilon})...")
        
        if not self.target_col:
            print("Error: This method requires a target column.")
            return []
        
        # Pre-compute neighbor graph
        neighbor_graph = build_neighbor_graph(self.df, self.feature_cols, max_distance)
        target_array = self.df[self.target_col].values
        
        groups = []
        skipped_count = 0
        skipped_reasons = {
            'insufficient_neighbors': 0,
            'build_failed': 0,
            'balance_too_high': 0
        }
        
        for center_idx in range(len(self.df)):
            # Get all neighbors within max_distance (including center itself)
            neighbors = list(neighbor_graph[center_idx])
            
            if len(neighbors) < min_group_size:
                skipped_count += 1
                skipped_reasons['insufficient_neighbors'] += 1
                continue
            
            center_target = target_array[center_idx]
            max_distance 
            # Build a balanced subset, growing from min to max size
            subset = self._build_balanced_subset_growing(
                center_idx, neighbors, center_target, target_array, 
                min_group_size, max_group_size, epsilon
            )
            
            if subset is None:
                skipped_count += 1
                skipped_reasons['build_failed'] += 1
                continue
            
            # Final validation (should already be < epsilon from the growing process)
            same_count = sum(1 for idx in subset if target_array[idx] == center_target)
            diff_count = len(subset) - same_count
            balance = abs(same_count - diff_count) / len(subset)
            
            if balance >= epsilon:
                skipped_count += 1
                skipped_reasons['balance_too_high'] += 1
                continue
            
            # Create hamming ball
            varying_features = identify_varying_features(self.df, subset, self.feature_cols)
            
            group = HammingBall(
                group_id=len(groups),
                reference_idx=center_idx,
                counterfactual_indices=subset,
                max_distance=max_distance,
                feature_names=varying_features
            )
            
            groups.append(group)
            
            if (len(groups) + skipped_count) % 50 == 0:
                print(f"Processed {len(groups) + skipped_count}/{len(self.df)} points "
                      f"({len(groups)} valid groups, {skipped_count} skipped)...")
        
        self.hamming_groups = groups
        
        print(f"\nFound {len(groups)} hamming balls (one per valid center point)")
        print(f"Skipped {skipped_count} points total:")
        print(f"  - Insufficient neighbors (< {min_group_size}): {skipped_reasons['insufficient_neighbors']}")
        print(f"  - Could not build balanced subset: {skipped_reasons['build_failed']}")
        print(f"  - Balance too high (>= {epsilon}): {skipped_reasons['balance_too_high']}")
        print(f"Coverage: {len(groups)}/{len(self.df)} = {len(groups)/len(self.df)*100:.1f}%")
        
        if groups:
            # Calculate balance and size statistics
            balances = []
            sizes = []
            for g in groups:
                center_target = target_array[g.reference_idx]
                same_count = sum(1 for idx in g.counterfactual_indices 
                               if target_array[idx] == center_target)
                diff_count = len(g.counterfactual_indices) - same_count
                balance = abs(same_count - diff_count) / len(g.counterfactual_indices)
                balances.append(balance)
                sizes.append(len(g.counterfactual_indices))
            
            print(f"Balance statistics - Mean: {np.mean(balances):.3f}, "
                  f"Max: {np.max(balances):.3f}, "
                  f"Min: {np.min(balances):.3f}")
            print(f"Size statistics - Mean: {np.mean(sizes):.1f}, "
                  f"Max: {max(sizes)}, "
                  f"Min: {min(sizes)}, "
                  f"Median: {np.median(sizes):.1f}")
        
        return groups
    
    def _build_balanced_subset_growing(self,center_idx, neighbors,
                                        center_target, target_array,
                                        min_group_size, max_group_size,
                                        epsilon):
        """
        Build a balanced subset that grows from min_group_size to max_group_size.
        
        Strategy:
        1. Build initial subset of min_group_size using greedy alternating approach
        2. If balance < epsilon, keep adding points greedily
        3. After each addition, check if balance is still < epsilon
        4. Stop when balance >= epsilon (return last valid subset) or max_group_size reached
        
        Args:
            center_idx: Index of center point
            neighbors: List of neighbor indices (including center)
            center_target: Target value of center point
            target_array: Array of all target values
            min_group_size: Minimum subset size
            max_group_size: Maximum subset size
            epsilon: Maximum allowed balance parameter
            
        Returns:
            List of indices forming the subset, or None if can't build valid subset
        """

        # Separate neighbors by target value
        same_target = [idx for idx in neighbors if target_array[idx] == center_target]
        diff_target = [idx for idx in neighbors if target_array[idx] != center_target]

        # Start with center
        subset = [center_idx]
        
        # Remove center from available pools
        if center_idx in same_target:
            same_target = [idx for idx in same_target if idx != center_idx]
        
        # Alternate between adding same and different target values
        add_same = False  # Start by adding different target (to balance the center)
        
        # First, build to min_group_size
        while len(subset) < min_group_size:
            if add_same:
                candidates = same_target
            else:
                candidates = diff_target
            
            if not candidates:
                # Try the other pool if current pool is empty
                candidates = diff_target if add_same else same_target
                
            if not candidates:
                # Can't add more points
                return None
            
            # Randomly select from candidates
            selected_idx = np.random.choice(len(candidates))
            selected = candidates[selected_idx]
            
            subset.append(selected)
            
            # Remove selected from both pools
            same_target = [idx for idx in same_target if idx != selected]
            diff_target = [idx for idx in diff_target if idx != selected]
            
            # Alternate
            add_same = not add_same
        
        # Check if min subset meets epsilon constraint
        same_count = sum(1 for idx in subset if target_array[idx] == center_target)
        diff_count = len(subset) - same_count
        balance = abs(same_count - diff_count) / len(subset)

        if balance >= epsilon:
            return None
        
        # Now grow beyond min_group_size up to max_group_size
        last_valid_subset = subset.copy()
        
        while len(subset) < max_group_size:
            if add_same:
                candidates = same_target
            else:
                candidates = diff_target
            
            if not candidates:
                # Try the other pool if current pool is empty
                candidates = diff_target if add_same else same_target
                
            if not candidates:
                # No more points to add
                break
            
            # Randomly select from candidates
            selected_idx = np.random.choice(len(candidates))
            selected = candidates[selected_idx]
            
            subset.append(selected)
            
            # Check balance
            same_count = sum(1 for idx in subset if target_array[idx] == center_target)
            diff_count = len(subset) - same_count
            balance = abs(same_count - diff_count) / len(subset)

            if balance >= epsilon:
                # Balance violated, return last valid subset
                return last_valid_subset
            
            # Balance still good, update last valid
            last_valid_subset = subset.copy()
            
            # Remove selected from both pools
            same_target = [idx for idx in same_target if idx != selected]
            diff_target = [idx for idx in diff_target if idx != selected]
            
            # Alternate
            add_same = not add_same
        
        return last_valid_subset


In [ ]:
# utils

def check_hamming_ball_balance(idx, neighbor_graph, target_array):
    """
    Returns the targets of all neighbours in your hamming ball 

    Return:
    Literal [0,1]
    """

    targets = []

    for neighbour_idx in neighbor_graph[idx]:
        targets.append(target_array[neighbour_idx])

    num_neighbours = len(neighbor_graph[idx])

    num_1s = int(sum(targets))
    num_0s = num_neighbours-num_1s

    return num_1s, num_0s
 

In [ ]:
from dataclasses import dataclass, asdict
from typing import Optional, List, Dict, Any, Type
from tqdm import tqdm
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from pathlib import Path
from src.templates.heart_disease import HeartDisease
from src.templates.pima_diabetes import PimaDiabetes
from src.templates.breast_cancer_recurrence import BreastCancerRecurrence
from src.templates.multiple_choice_dataset import MultipleChoiceDataset
from src.templates.trait import Trait
from src.templates.income import IncomeDataset
from src.templates.attrition import AttritionDataset
from src.templates.moral_machines import MoralMachines
from src.templates.bank_marketing import BankMarketing
from src.templates.bbq_dataset import BBQDataset
import random

@dataclass
class ModelInfo:
    """
    Information about a model used for generation.
    """
    model: Optional[str] = None                 # Name of the model
    temperature: Optional[float] = None         # Temperature used for generation
    max_tokens: Optional[int] = None            # Max tokens used for generation
    thinking: Optional[str] = None             # Whether this is a thinking/reasoning model variant
    seed: Optional[int] = None                  # Random seed
    additional_params: Optional[dict] = None    # Additional sampling params, e.g. top_k, min_p

# Backwards compatibility aliases
ReferenceModelInfo = ModelInfo
PredictorModelInfo = ModelInfo

@dataclass
class Response:
    """
    Encapsulates a model's response with metadata.
    """
    cot: Optional[str] = None                                   # chain-of-thought for reasoning models
    raw_response: Optional[str] = None                          # Raw LLM output (JSON string)
    parsed_response: Optional[Dict[str, Any]] = None            # Parsed JSON dict
    answer: Optional[str] = None                                # Extracted answer field from parsed response
    model_info: Optional[ModelInfo] = None                      # Model that generated this response
    predictor_answers: Optional[List[Optional[str]]] = None     # All predictor answers from all models/repeats
    predictor_names: Optional[List[str]] = None                 # Model name for each answer in predictor_answers
    input_tokens: Optional[int] = None                          # Number of input tokens
    reasoning_tokens: Optional[int] = None                      # Number of reasoning (CoT) tokens
    output_tokens: Optional[int] = None                         # Number of output tokens (including reasoning)

@dataclass
class OriginalQuestion:
    dataset: str                                        # Name of the dataset (e.g., 'SQuAD', 'TruthfulQA'). 
    question: str                                       # The original question text (unformatted - just the natural language question)          
    question_prompt: str                                # The original question text (**full prompt**) 
    question_idx: int                                   # Unique index of the question within the dataset.
    ground_truth: Optional[str] = None                  # Ground-truth answer, if available.
    answer_first: Optional[bool] = None                 # Whether the original question asks for the answer first or answer second (after explanation)
    description: Optional[str] = None                   # Natural language description (e.g., "This is a male patient, 50-60 years old...")
    question_options: Optional[dict] = None             # Multiple-choice options, if applicable. Keys are A,B,C, D, values are options.
    reference_response: Optional[Response] = None       # Reference model's response to this original question (with explanation)


@dataclass
class CounterfactualInfo:
    generator_model: str                                # Name of the model that generated the counterfactual.
    generator_method: str                               # Method used to generate the counterfactual (e.g., 'Matton').
    question: str                                       # The counterfactual question text (unformatted - just the natural language question)
    question_prompt: str                                # The counterfactual question text (**full prompt**)
    generator_model_info: Optional[ModelInfo] = None    # Metadata about the generator model.
    generator_model_cot: Optional[str] = None           # Save the CoT from the counterfactual generator model. Useful for debugging.
    generator_model_raw: Optional[str] = None           # Raw generator output, if available.
    question_idx: Optional[int] = None                  # ← optional, auto-assigned later
    ground_truth: Optional[str] = None
    description: Optional[str] = None                   # Natural language description of counterfactual
    coherence_scored_by_generator: Optional[bool] = None
    coherence_explanation_by_generator: Optional[str] = None
    coherence_external_scoring_model: Optional[str] = None
    coherence_scored_by_external_model: Optional[bool] = None
    coherence_explanation_by_external_model: Optional[str] = None
    hamming_distance: Optional[int] = None
    question_options: Optional[dict] = None             # Multiple-choice options, if applicable. Keys are A,B,C, D, values are options.
    
    # Reference model's response to this counterfactual question (without explanation context)
    reference_response: Optional[Response] = None
    
    # Phase 3: Predictor prompts (generated using original question's reference answer)
    prompt_with_explanation: Optional[str] = None       # Prompt including original's explanation
    prompt_without_explanation: Optional[str] = None    # Prompt with only original's answer
    
    # Predictor model's responses to the prompts
    predictor_response_with_explanation: Optional[Response] = None
    predictor_response_without_explanation: Optional[Response] = None

    # Testability assessment (optional, scored 0-10)
    predictor_counterfactual_testability_score: Optional[float] = None
    predictor_counterfactual_testability_cot: Optional[str] = None
    
    # Cross-model experiment fields (only populated by cross_model_experiment.py)
    is_cross_model_explanation: Optional[bool] = None       # True if prompt uses explanation from different model
    explanation_source_model_info: Optional[ModelInfo] = None  # ModelInfo of the model that provided the explanation

@dataclass
class MatchInfo:
    """
    Evaluation metrics comparing predictor outputs against reference answers.
    """
    match_with_explanation: Optional[int] = None    # 1 if prediction (with explanation) matches reference answer, else 0.
    match_without_explanation: Optional[int] = None # 1 if prediction (without explanation) matches reference answer, else 0.
    match_delta: Optional[int] = None               # Difference: match_with_explanation - match_without_explanation.

@dataclass
class FaithfulnessRecord:
    """
    Represents one row in the counterfactual faithfulness/simulatability dataset.
    """
    original_question: OriginalQuestion
    counterfactual: CounterfactualInfo
    match_info: Optional[MatchInfo] = None

    def to_flat_dict(self):
        """Flatten the record for DataFrame/Parquet storage."""
        flat = {}
        
        # Flatten OriginalQuestion to add original_ prefix.
        if self.original_question is not None:
            for k, v in asdict(self.original_question).items():
                if k == 'reference_response' and isinstance(v, dict):
                    # Flatten Response object
                    for response_key, response_val in v.items():
                        if response_key == 'model_info' and isinstance(response_val, dict):
                            # Flatten ModelInfo
                            for model_key, model_val in response_val.items():
                                flat[f"original_reference_response_model_info_{model_key}"] = model_val
                        else:
                            flat[f"original_reference_response_{response_key}"] = response_val
                else:
                    flat[f"original_{k}"] = v
        
        # Flatten CounterfactualInfo
        if self.counterfactual is not None:
            for k, v in asdict(self.counterfactual).items():
                if k == 'generator_model_info' and isinstance(v, dict):
                    for model_key, model_val in v.items():
                        flat[f"counterfactual_generator_model_info_{model_key}"] = model_val
                    continue
                if k == 'explanation_source_model_info' and isinstance(v, dict):
                    for model_key, model_val in v.items():
                        flat[f"counterfactual_explanation_source_model_info_{model_key}"] = model_val
                    continue
                if k in ['reference_response', 'predictor_response_with_explanation', 'predictor_response_without_explanation'] and isinstance(v, dict):
                    # Flatten Response object
                    for response_key, response_val in v.items():
                        if response_key == 'model_info' and isinstance(response_val, dict):
                            # Flatten ModelInfo
                            for model_key, model_val in response_val.items():
                                flat[f"counterfactual_{k}_model_info_{model_key}"] = model_val
                        else:
                            flat[f"counterfactual_{k}_{response_key}"] = response_val
                else:
                    flat[f"counterfactual_{k}"] = v
        
        # Flatten MatchInfo
        if self.match_info is not None:
            for k, v in asdict(self.match_info).items():
                flat[f"match_{k}"] = v
        
        return flat


# -------------------------------------------------------------------------
# CounterfactualDatabase
# -------------------------------------------------------------------------


class CounterfactualDatabase:
    """
    A manager for FaithfulnessRecord objects.

    Handles:
        - Automatic index assignment.
        - Adding new records.
        - Saving/loading to Parquet.
        - Conversion to/from DataFrames.
    """

    dataset_class_map = {
            'heart_disease': HeartDisease,
            'pima_diabetes': PimaDiabetes,
            'breast_cancer_recurrence': BreastCancerRecurrence,
            'trait': Trait,
            'multiple_choice_dataset': MultipleChoiceDataset,
            'income': IncomeDataset,
            'attrition': AttritionDataset,
            'moral_machines': MoralMachines,
            'bank_marketing': BankMarketing,
            'bbq': BBQDataset,
        }
    
    def __init__(self):
        self.records: List[FaithfulnessRecord] = []
    # --------------------------------------------------

    def add_record(self, record: FaithfulnessRecord) -> None:
        """Add a new record and automatically assign indices."""

        # Assign unique cf_question index
        cf_question_indices = set(
            r.counterfactual.question_idx for r in self.records
            if r.counterfactual.question_idx is not None
        )
        max_index = max(cf_question_indices) if cf_question_indices else 100000000
        if record.counterfactual.question_idx is None:
            record.counterfactual.question_idx = max_index + 1
        self.records.append(record)

    # --------------------------------------------------

    def to_dataframe(self) -> pd.DataFrame:
        """Convert the full database to a pandas DataFrame."""
        return pd.DataFrame([r.to_flat_dict() for r in self.records])

    # --------------------------------------------------

    def save_parquet(self, path: str | Path) -> None:
        """Save the entire database to a Parquet file."""
        df = self.to_dataframe()
        pq.write_table(pa.Table.from_pandas(df), path)

    # --------------------------------------------------
    @classmethod
    def load_parquet(cls, path: str | Path) -> "CounterfactualDatabase":
        """Load a CounterfactualDatabase from a Parquet file."""
        from .schema import (
            FaithfulnessRecord,
            OriginalQuestion,
            CounterfactualInfo,
            MatchInfo,
            Response,
            ModelInfo,
        )

        df = pq.read_table(path).to_pandas()
        db = cls()
        
        # Convert to list of dicts - much faster than iterrows()
        rows = df.to_dict('records')
        db.records = [None] * len(rows)  # Pre-allocate
        
        for i, row in enumerate(tqdm(rows, desc="Loading records")):
            # Reconstruct Response objects
            def reconstruct_response(prefix):
                """Reconstruct a Response object from flattened columns."""
                response_fields = {}
                model_info_fields = {}
                
                for k, v in row.items():
                    if k.startswith(f"{prefix}_model_info_"):
                        model_key = k[len(f"{prefix}_model_info_"):]
                        model_info_fields[model_key] = v
                    elif k.startswith(f"{prefix}_"):
                        response_key = k[len(f"{prefix}_"):]
                        if response_key != 'model_info':
                            response_fields[response_key] = v
                
                if response_fields:
                    response_fields['model_info'] = ModelInfo(**model_info_fields) if model_info_fields else None
                    return Response(**response_fields)
                return None
            
            # Reconstruct OriginalQuestion
            original_fields = {}
            for k, v in row.items():
                if k.startswith("original_") and not k.startswith("original_reference_response_"):
                    original_fields[k[len("original_"):]] = v
            original_fields['reference_response'] = reconstruct_response("original_reference_response")
            original_question = OriginalQuestion(**original_fields)
            
            # Reconstruct CounterfactualInfo
            counterfactual_fields = {}
            for k, v in row.items():
                if k.startswith("counterfactual_generator_model_info_"):
                    continue  # handled separately
                if k.startswith("counterfactual_explanation_source_model_info_"):
                    continue  # handled separately
                if k.startswith("counterfactual_") and not any(
                    k.startswith(f"counterfactual_{resp}_") 
                    for resp in ['reference_response', 'predictor_response_with_explanation', 'predictor_response_without_explanation']
                ):
                    counterfactual_fields[k[len("counterfactual_"):]] = v
            # Rebuild generator_model_info
            generator_model_info_fields = {}
            for k, v in row.items():
                if k.startswith("counterfactual_generator_model_info_"):
                    model_key = k[len("counterfactual_generator_model_info_"):]
                    generator_model_info_fields[model_key] = v
            counterfactual_fields['generator_model_info'] = ModelInfo(**generator_model_info_fields) if generator_model_info_fields else None
            # Rebuild explanation_source_model_info
            explanation_source_model_info_fields = {}
            for k, v in row.items():
                if k.startswith("counterfactual_explanation_source_model_info_"):
                    model_key = k[len("counterfactual_explanation_source_model_info_"):]
                    explanation_source_model_info_fields[model_key] = v
            counterfactual_fields['explanation_source_model_info'] = ModelInfo(**explanation_source_model_info_fields) if explanation_source_model_info_fields else None
            counterfactual_fields['reference_response'] = reconstruct_response("counterfactual_reference_response")
            counterfactual_fields['predictor_response_with_explanation'] = reconstruct_response("counterfactual_predictor_response_with_explanation")
            counterfactual_fields['predictor_response_without_explanation'] = reconstruct_response("counterfactual_predictor_response_without_explanation")
            counterfactual = CounterfactualInfo(**counterfactual_fields)
            
            # Reconstruct MatchInfo
            match_fields = {}
            for k, v in row.items():
                if k.startswith("match_"):
                    match_fields[k[len("match_"):]] = v
            match_info = MatchInfo(**match_fields) if match_fields else None

            record = FaithfulnessRecord(
                original_question=original_question,
                counterfactual=counterfactual,
                match_info=match_info,
            )
            db.records[i] = record
        return db

In [ ]:
dataset = HeartDisease
df = HeartDisease.load_dataset()

converter = TabularToTextConverter(
    df, 
    target_col='target',
    description_generator=dataset.description_generator,
    prompt_generator=dataset.create_reference_prompt,
    dataset_name=dataset.to_string(),
    target_formatter=dataset.format_target  # Pass the format_target method
)

answer_first_only = False
output_dir = "."
groups = converter.find_hamming_balls_repeated(max_distance=2, min_group_size=5,max_group_size=10,epsilon=0.3)
output_file = os.path.join(output_dir, f'{dataset.to_string()}_counterfactual_dataset_balanced.parquet')

db = CounterfactualDatabase()



Loading UCI Heart Disease dataset...
Loaded 264 samples with 10 features
Removed 33 duplicate rows created by binning

Feature value counts:
  sex: 2 unique values
  cp: 4 unique values
  fbs: 2 unique values
  restecg: 3 unique values
  exang: 2 unique values
  slope: 3 unique values
  age_group: 4 unique values
  chol_level: 3 unique values
  trestbps_level: 3 unique values
  target: 2 unique values
Initialized with 264 rows and 9 features
Features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'age_group', 'chol_level', 'trestbps_level']

Finding repeated hamming balls (max_distance=2, min_size=5, max_size=10, epsilon=0.3)...
Pre-computing neighbor graph for 264 points...
Neighbor graph built. Average neighbors per point: 14.6
Processed 100/264 points (73 valid groups, 27 skipped)...
Processed 150/264 points (111 valid groups, 39 skipped)...

Found 191 hamming balls (one per valid center point)
Skipped 73 points total:
  - Insufficient neighbors (< 5): 23
  - Could not build bal

In [ ]:
data.shape 
# (264, 10)

x = data[:, np.newaxis, :]
x.shape
# (264, 1, 10)

y = data[np.newaxis, :, :]
y.shape
# (1, 264, 10)

distances = differences.sum(axis=2)
distances.shape
# (264, 264)

differences.shape

differences

distances

array([[0, 6, 6, ..., 5, 9, 7],
       [6, 0, 2, ..., 4, 4, 6],
       [6, 2, 0, ..., 5, 4, 5],
       ...,
       [5, 4, 5, ..., 0, 4, 7],
       [9, 4, 4, ..., 4, 0, 5],
       [7, 6, 5, ..., 7, 5, 0]], shape=(264, 264))

In [ ]:
# Roughly tracing the find_hamming_balls_repeated_func

df = HeartDisease.load_dataset()

feature_cols = [col for col in df.columns if col != 'target']
max_distance = 2
distances = compute_all_pairwise_distances(df, feature_cols, max_distance)
neighbor_graph = build_neighbor_graph(df, feature_cols, max_distance)
target_array = df['target'].values
subsets = []
groups = []

for center_idx in range(len(df)):
    subset = _build_balanced_subset_growing(
        center_idx = center_idx, neighbors=neighbor_graph[center_idx], center_target=target_array[center_idx],
        target_array=target_array , min_group_size=5, max_group_size=10, epsilon = 0.3
    )

    # def identify varying features
    if subset == None:
        continue

    varying_features = identify_varying_features(df, subset, feature_cols)

    group = HammingBall(
                group_id=len(groups),
                reference_idx=center_idx,
                counterfactual_indices=subset,
                max_distance=max_distance,
                feature_names=varying_features
    )

    groups.append(group)

hamming_groups = groups

sizes = []
balances = []

for g in groups:
    center_target = target_array[g.reference_idx]
    same_count = sum(1 for idx in g.counterfactual_indices 
                    if target_array[idx] == center_target)
    diff_count = len(g.counterfactual_indices) - same_count
    balance = abs(same_count - diff_count) / len(g.counterfactual_indices)
    balances.append(balance)
    sizes.append(len(g.counterfactual_indices))

print(
    f"Balance statistics - Mean: {np.mean(balances):.3f}, "
    f"Max: {np.max(balances):.3f}, "
    f"Min: {np.min(balances):.3f}"
)
print(
    f"Size statistics - Mean: {np.mean(sizes):.1f}, "
    f"Max: {max(sizes)}, "
    f"Min: {min(sizes)}, "
    f"Median: {np.median(sizes):.1f}"
)


# subsets = [x for x in subsets if x]

# create hamming ball


Loading UCI Heart Disease dataset...
Loaded 264 samples with 10 features
Removed 33 duplicate rows created by binning

Feature value counts:
  sex: 2 unique values
  cp: 4 unique values
  fbs: 2 unique values
  restecg: 3 unique values
  exang: 2 unique values
  slope: 3 unique values
  age_group: 4 unique values
  chol_level: 3 unique values
  trestbps_level: 3 unique values
  target: 2 unique values
Pre-computing neighbor graph for 264 points...
Neighbor graph built. Average neighbors per point: 14.6
Balance statistics - Mean: 0.088, Max: 0.250, Min: 0.000
Size statistics - Mean: 8.9, Max: 10, Min: 5, Median: 10.0


In [109]:
len(hamming_groups)

191

In [ ]:
subsets = [x for x in subsets if x]
# subset = [1, np.int64(222), np.int64(101), np.int64(84), np.int64(55)]
# len = 54

54

In [ ]:
for idx in range(10):
    num_1s,num_0s = check_hamming_ball_balance(idx = idx, neighbor_graph=neighbor_graph, target_array=target_array)
    print(f"{idx=} : {num_1s=},{num_0s=}")
    

idx=0 : num_1s=0,num_0s=7
idx=1 : num_1s=53,num_0s=2
idx=2 : num_1s=35,num_0s=5
idx=3 : num_1s=1,num_0s=16
idx=4 : num_1s=1,num_0s=37
idx=5 : num_1s=6,num_0s=45
idx=6 : num_1s=18,num_0s=5
idx=7 : num_1s=5,num_0s=18
idx=8 : num_1s=50,num_0s=7
idx=9 : num_1s=16,num_0s=0


In [107]:
feature_cols

['sex',
 'cp',
 'fbs',
 'restecg',
 'exang',
 'slope',
 'age_group',
 'chol_level',
 'trestbps_level']

In [92]:
varying_features

['cp', 'slope', 'age_group', 'chol_level', 'trestbps_level', 'target']